# Qwen3-14B Zero-shot Writing Scoring Baseline

목표: Google Colab에서 `Qwen/Qwen3-14B`를 내려받아 별도 학습, 점수 보정, fallback 없이 train/validation 데이터에 대한 기준선 점수를 확인한다.

성공 기준:
- 데이터셋 폴더의 train/validation JSONL을 읽는다.
- Qwen3-14B 4bit 추론으로 세 영역 점수와 근거 JSON을 생성한다.
- 파싱 성공률, RMSE, Spearman을 train/validation 각각 산출한다.
- 생성 원문과 파싱 결과를 JSONL로 저장해 중간에 끊겨도 이어서 실행할 수 있다.


## 1. Colab 런타임 준비

권장 런타임은 A100급 GPU다. L4/T4에서도 4bit 추론은 시도할 수 있지만 속도와 메모리 여유가 작을 수 있다.

아래 셀은 Colab에서 필요한 패키지를 설치한다. 로컬에서 구조만 확인할 때는 실행하지 않아도 된다.


In [ ]:
# Colab dependency install
%pip -q install -U "transformers>=4.51.0" accelerate bitsandbytes pandas scipy tqdm sentencepiece


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 136.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 8.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which

In [ ]:
# GPU and disk sanity check
import os
import shutil
import subprocess
from pathlib import Path

try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as exc:
    print(f"nvidia-smi unavailable: {exc}")

for path in ["/content", str(Path.cwd())]:
    if Path(path).exists():
        total, used, free = shutil.disk_usage(path)
        print(path, {"total_gb": round(total / 1e9, 1), "free_gb": round(free / 1e9, 1)})


Fri Jul  3 04:36:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   27C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# Optional: mount Google Drive if your project folder is stored there.
MOUNT_GOOGLE_DRIVE = False

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ModuleNotFoundError:
        print("google.colab is not available outside Colab; skipping Drive mount.")


## 2. 설정

`PROJECT_ROOT`는 `데이터셋` 폴더를 포함하는 프로젝트 루트다. Colab Drive를 쓰는 경우 Drive를 마운트한 뒤 경로 후보에 자동으로 잡히는지 확인한다.

전체 점수를 보려면 `MAX_TRAIN_ROWS=None`, `MAX_VALIDATION_ROWS=None`을 유지한다. 먼저 동작 확인만 하려면 각각 5~20 정도로 줄인다.


In [ ]:
from __future__ import annotations

import json
import math
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

MODEL_ID = "Qwen/Qwen3-14B"
RUN_TAG = "qwen3_14b_zero_shot_strict_v2"

# Full run by default. Set small integers for a smoke test.
MAX_TRAIN_ROWS = None          # train has about 2,000 rows
MAX_VALIDATION_ROWS = None     # validation has about 400 rows

MAX_INPUT_TOKENS = 4096
MAX_NEW_TOKENS = 768
GENERATION_BATCH_SIZE = 30

DOMAINS = ["content", "organization", "expression"]

CANDIDATE_ROOTS = [
    Path.cwd(),
    Path("/content/drive/MyDrive/글쓰기 채점 능력 평가"),
    Path("/content/글쓰기 채점 능력 평가"),
]

PROJECT_ROOT = next((p for p in CANDIDATE_ROOTS if (p / "데이터셋").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("데이터셋 폴더를 찾지 못했습니다. PROJECT_ROOT를 직접 지정하세요.")

DATA_DIR = PROJECT_ROOT / "데이터셋"
TRAIN_PATH = DATA_DIR / "글쓰기채점능력평가2026_train.jsonl"
VALIDATION_PATH = DATA_DIR / "글쓰기채점능력평가2026_validation.jsonl"
OUTPUT_DIR = PROJECT_ROOT / "output" / "qwen3_14b_zero_shot"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({
    "project_root": str(PROJECT_ROOT),
    "train_path_exists": TRAIN_PATH.exists(),
    "validation_path_exists": VALIDATION_PATH.exists(),
    "output_dir": str(OUTPUT_DIR),
})


{'project_root': '/content', 'train_path_exists': True, 'validation_path_exists': True, 'output_dir': '/content/output/qwen3_14b_zero_shot'}


## 3. 데이터 로드

데이터는 JSONL이며, 각 행은 `prompt`, `essay`, `score`를 포함한다. `score`에는 세 영역과 평균 점수가 들어 있다.


In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def to_dataframe(rows: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    for domain in DOMAINS:
        df[f"gold_{domain}"] = df["score"].apply(lambda x: float(x[domain]))
    df["gold_average"] = df["score"].apply(lambda x: float(x.get("average", np.mean([x[d] for d in DOMAINS]))))
    return df

train_df = to_dataframe(read_jsonl(TRAIN_PATH))
validation_df = to_dataframe(read_jsonl(VALIDATION_PATH))

print("train", train_df.shape)
print("validation", validation_df.shape)
display(train_df[["id", "prompt_num", "gold_content", "gold_organization", "gold_expression", "gold_average"]].head())


train (2000, 10)
validation (400, 10)


,id,prompt_num,gold_content,gold_organization,gold_expression,gold_average
0,GWGR2300001070,Q1,3.5,4.25,3.25,3.67
1,GWGR2300001100,Q2,4.2,4.75,4.25,4.40
2,GWGR2300001190,Q1,4.3,4.25,4.50,4.35
3,GWGR2300001320,Q2,3.5,2.75,3.50,3.25
4,GWGR2300001770,Q3,4.0,3.25,3.75,3.67


In [ ]:
def score_distribution(df: pd.DataFrame, name: str) -> pd.DataFrame:
    rows = []
    for col in ["gold_content", "gold_organization", "gold_expression", "gold_average"]:
        rows.append({
            "split": name,
            "target": col,
            "count": len(df),
            "mean": df[col].mean(),
            "std": df[col].std(),
            "min": df[col].min(),
            "max": df[col].max(),
        })
    return pd.DataFrame(rows)

display(pd.concat([
    score_distribution(train_df, "train"),
    score_distribution(validation_df, "validation"),
], ignore_index=True).round(4))


,split,target,count,mean,std,min,max
0,train,gold_content,2000,3.2778,0.6455,1.00,5.0
1,train,gold_organization,2000,3.3374,0.8358,1.00,5.0
2,train,gold_expression,2000,3.6726,0.6801,1.25,5.0
3,train,gold_average,2000,3.4292,0.6297,1.15,5.0
4,validation,gold_content,400,3.2290,0.6538,1.10,4.7
5,validation,gold_organization,400,3.2869,0.8711,1.00,5.0
6,validation,gold_expression,400,3.6769,0.6810,1.25,5.0
7,validation,gold_average,400,3.3978,0.6542,1.29,4.9


## 4. 모델 로드

이 기준선은 학습, calibration, 규칙 기반 fallback을 하지 않는다. 모델이 생성한 텍스트에서 JSON만 파싱하고, 파싱 실패는 실패로 기록한다.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU가 필요합니다. Colab 런타임을 GPU로 변경하세요.")

major, minor = torch.cuda.get_device_capability(0)
compute_dtype = torch.bfloat16 if major >= 8 else torch.float16
print("compute_dtype", compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
    trust_remote_code=False,
)
model.eval()
print("loaded", MODEL_ID)


compute_dtype torch.bfloat16


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

loaded Qwen/Qwen3-14B


## 5. 프롬프트와 파서

프롬프트는 제출 형식에 맞춰 JSON만 요구한다. 점수는 1~5 범위의 숫자로 받는다. 여기서는 점수 보정이나 fallback을 하지 않으므로, 구조가 깨지면 그대로 실패로 계산한다.


In [ ]:
SYSTEM_PROMPT = '''너는 한국어 논증적 글을 일관되게 직접 채점하는 평가자이다.
essay를 읽고 content, organization, expression 세 기준을 모두 평가하라.
반드시 JSON 객체만 출력하고, Markdown 코드블록이나 추가 설명은 출력하지 마라.
생각 과정은 출력하지 마라.
5점은 매우 우수한 글에만 부여하고, 일반적으로 잘 쓴 글은 3~4점으로 평가하라.
근거가 일반적이거나 구체성이 부족하면 content를 낮추고, 구조는 있으나 연결과 전개가 약하면 organization을 낮추며, 문법·어휘·문장 흐름 문제가 보이면 expression을 낮춰라.
관대하게 점수를 올리지 말고, 감점 사유를 먼저 확인한 뒤 종합 판단하라.
각 score는 1 이상 5 이하의 숫자여야 한다.
rationale은 실제 essay 내용에 근거해 한국어 1~2문장으로 작성하라.
'''.strip()


def build_user_prompt(row: pd.Series) -> str:
    return f'''[평가 기준]
content: 주제 이해, 입장 명확성, 주장과 근거의 타당성, 논증의 깊이
organization: 서론-본론-결론 구조, 문단 구성, 전개 순서, 연결성
expression: 문장 정확성, 어휘 선택, 문법, 표현의 자연스러움

[점수 앵커]
5: 결함이 거의 없고, 주장·근거·구성·표현이 모두 매우 우수함
4: 대체로 우수하지만 일부 근거의 구체성, 연결, 표현상 아쉬움이 있음
3: 기본 요건은 충족하지만 근거가 일반적이거나 전개·표현의 약점이 뚜렷함
2: 입장, 근거, 구성, 표현 중 여러 요소가 부족해 설득력이 낮음
1: 주제 이해, 논증 구조, 문장 표현이 전반적으로 매우 미흡함

[출력 형식]
{{
  "content": {{"score": 1, "rationale": "content 판단 근거"}},
  "organization": {{"score": 1, "rationale": "organization 판단 근거"}},
  "expression": {{"score": 1, "rationale": "expression 판단 근거"}}
}}

[prompt_text]
{row['prompt']}

[essay_text]
{row['essay']}
'''.strip()


def apply_chat_template_no_thinking(messages: list[dict]) -> str:
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


def make_prompt(row: pd.Series) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(row)},
    ]
    return apply_chat_template_no_thinking(messages)


In [ ]:
def repair_common_json_errors(text: str) -> str:
    first = text.find("{")
    last = text.rfind("}")
    candidate = text[first:last + 1] if first >= 0 and last > first else text

    # Common Qwen typo: rationale string closes with ")" instead of "}"
    candidate = re.sub(
        r'("rationale"\s*:\s*"[^"\\]*(?:\\.[^"\\]*)*")\s*\)',
        r"\1}",
        candidate,
        flags=re.S,
    )

    # Remove trailing commas before object/list close
    candidate = re.sub(r",\s*([}\]])", r"\1", candidate)
    return candidate


def extract_first_json_object(text: str):
    decoder = json.JSONDecoder()

    repaired = repair_common_json_errors(text)
    try:
        return decoder.raw_decode(repaired[repaired.find("{"):])[0]
    except Exception:
        pass

    # Fallback: scan every opening brace, but only accept full schema later.
    for i, ch in enumerate(repaired):
        if ch != "{":
            continue
        try:
            obj, _ = decoder.raw_decode(repaired[i:])
            if isinstance(obj, dict) and all(d in obj for d in DOMAINS):
                return obj
        except json.JSONDecodeError:
            continue

    raise ValueError("No valid JSON object found")


def extract_domain_by_regex(text: str, domain: str) -> dict:
    m = re.search(rf'"{domain}"\s*:\s*\{{', text)
    if not m:
        raise ValueError(f"missing object: {domain}")

    body = text[m.end():]
    next_domain = re.search(r',\s*"(content|organization|expression)"\s*:', body)
    if next_domain:
        body = body[:next_domain.start()]

    score_m = re.search(r'"score"\s*:\s*"?([1-5](?:\.\d+)?)', body)
    rationale_m = re.search(r'"rationale"\s*:\s*"((?:\\.|[^"\\])*)"', body, flags=re.S)

    if not score_m:
        raise ValueError(f"missing score: {domain}")
    if not rationale_m:
        raise ValueError(f"missing rationale: {domain}")

    rationale_raw = rationale_m.group(1)
    try:
        rationale = json.loads(f'"{rationale_raw}"')
    except Exception:
        rationale = rationale_raw.replace('\\"', '"').strip()

    return {
        "score": float(score_m.group(1)),
        "rationale": rationale,
    }


def validate_prediction_object(obj: dict) -> dict:
    if not isinstance(obj, dict):
        raise ValueError("prediction is not a JSON object")

    clean = {}
    for domain in DOMAINS:
        if domain not in obj or not isinstance(obj[domain], dict):
            raise ValueError(f"missing object: {domain}")

        score = obj[domain].get("score")
        rationale = obj[domain].get("rationale")

        if isinstance(score, str):
            score_m = re.search(r"[1-5](?:\.\d+)?", score)
            if not score_m:
                raise ValueError(f"non-numeric score: {domain}")
            score = float(score_m.group(0))

        if not isinstance(score, (int, float)):
            raise ValueError(f"non-numeric score: {domain}")

        score = float(score)
        if not (1.0 <= score <= 5.0):
            raise ValueError(f"score out of range: {domain}={score}")

        if not isinstance(rationale, str) or not rationale.strip():
            raise ValueError(f"empty rationale: {domain}")

        clean[domain] = {
            "score": score,
            "rationale": rationale.strip(),
        }

    return clean


def parse_model_output(text: str) -> dict:
    try:
        obj = extract_first_json_object(text)
        clean = validate_prediction_object(obj)
        return {"ok": True, "parsed": clean, "error": None}
    except Exception as first_exc:
        try:
            repaired_text = repair_common_json_errors(text)
            clean = {
                domain: extract_domain_by_regex(repaired_text, domain)
                for domain in DOMAINS
            }
            clean = validate_prediction_object(clean)
            return {"ok": True, "parsed": clean, "error": None}
        except Exception as second_exc:
            return {
                "ok": False,
                "parsed": None,
                "error": f"{first_exc}; regex fallback failed: {second_exc}",
            }

## 6. 단일 샘플 추론 확인

첫 샘플에서 출력 형식이 안정적인지 확인한다. 이 셀이 실패하면 전체 루프를 돌리지 말고 프롬프트나 `MAX_NEW_TOKENS`를 먼저 조정한다.


In [ ]:
@torch.inference_mode()
def generate_batch(rows: list[pd.Series]) -> list[str]:
    prompts = [make_prompt(row) for row in rows]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    prompt_width = inputs["input_ids"].shape[-1]
    outputs = []
    for i in range(len(prompts)):
        generated_ids = output_ids[i, prompt_width:]
        outputs.append(tokenizer.decode(generated_ids, skip_special_tokens=True).strip())
    return outputs


def generate_one(row: pd.Series) -> str:
    return generate_batch([row])[0]

sample_row = validation_df.iloc[0]
sample_output = generate_one(sample_row)
print(sample_output)
parse_model_output(sample_output)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{
  "content": {"score": 4, "rationale": "주제를 잘 이해하고 로봇세 도입의 필요성을 명확히 주장했으며, 실직자 지원과 기업 독점 견제라는 구체적인 근거를 제시했으나, 논증의 깊이와 추가적인 대안 제시는 부족하다."},
  "organization": {"score": 4, "rationale": "서론-본론-결론 구조를 갖추고 있으며, 논점이 잘 전개되었으나, 일부 문단 간 연결성이 약하고, 결론 부분에서 주장의 강조가 부족하다."},
  "expression": {"score": 3, "rationale": "전체적으로 문법과 어휘 사용은 적절하나, 표현의 자연스러움과 문장 흐름에 약간의 어색함이 있으며, 일부 중복된 표현이 반복된다."}
}


{'ok': True,
 'parsed': {'content': {'score': 4.0,
   'rationale': '주제를 잘 이해하고 로봇세 도입의 필요성을 명확히 주장했으며, 실직자 지원과 기업 독점 견제라는 구체적인 근거를 제시했으나, 논증의 깊이와 추가적인 대안 제시는 부족하다.'},
  'organization': {'score': 4.0,
   'rationale': '서론-본론-결론 구조를 갖추고 있으며, 논점이 잘 전개되었으나, 일부 문단 간 연결성이 약하고, 결론 부분에서 주장의 강조가 부족하다.'},
  'expression': {'score': 3.0,
   'rationale': '전체적으로 문법과 어휘 사용은 적절하나, 표현의 자연스러움과 문장 흐름에 약간의 어색함이 있으며, 일부 중복된 표현이 반복된다.'}},
 'error': None}

## 7. 전체/부분 실행 루프

각 레코드는 생성 직후 JSONL로 저장된다. Colab 세션이 끊기면 같은 셀을 다시 실행해 이미 저장된 `id`를 건너뛸 수 있다. GPU 여유가 있으면 `GENERATION_BATCH_SIZE`를 3 또는 4로 올려 한 번에 여러 글을 생성한다.


In [ ]:
def select_rows(df: pd.DataFrame, limit: int | None) -> pd.DataFrame:
    if limit is None:
        return df.copy()
    return df.head(limit).copy()


def prediction_path(split_name: str) -> Path:
    return OUTPUT_DIR / f"{RUN_TAG}_{split_name}.jsonl"


def load_prediction_records(path: Path) -> list[dict]:
    if not path.exists():
        return []
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def iter_row_batches(df: pd.DataFrame, batch_size: int):
    for start in range(0, len(df), batch_size):
        yield [row for _, row in df.iloc[start:start + batch_size].iterrows()]


def run_split(df: pd.DataFrame, split_name: str) -> list[dict]:
    path = prediction_path(split_name)
    existing = load_prediction_records(path)
    done_ids = {r["id"] for r in existing if r.get("run_tag") == RUN_TAG}
    rows = df[~df["id"].isin(done_ids)].copy()
    print({
        "split": split_name,
        "target_rows": len(df),
        "already_done": len(done_ids),
        "remaining": len(rows),
        "batch_size": GENERATION_BATCH_SIZE,
        "path": str(path),
    })

    with path.open("a", encoding="utf-8") as f:
        for batch_rows in tqdm(
            iter_row_batches(rows, GENERATION_BATCH_SIZE),
            total=math.ceil(len(rows) / GENERATION_BATCH_SIZE) if len(rows) else 0,
            desc=split_name,
        ):
            started = time.time()
            raw_outputs = generate_batch(batch_rows)
            elapsed = time.time() - started
            per_item_elapsed = elapsed / max(len(batch_rows), 1)

            for row, raw_output in zip(batch_rows, raw_outputs):
                parsed = parse_model_output(raw_output)
                record = {
                    "run_tag": RUN_TAG,
                    "model_id": MODEL_ID,
                    "split": split_name,
                    "id": row["id"],
                    "prompt_num": row.get("prompt_num"),
                    "gold_score": row["score"],
                    "raw_output": raw_output,
                    "parse_ok": parsed["ok"],
                    "parsed": parsed["parsed"],
                    "parse_error": parsed["error"],
                    "elapsed_sec": round(per_item_elapsed, 3),
                    "batch_size": len(batch_rows),
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
            f.flush()

    return load_prediction_records(path)

train_eval_df = select_rows(train_df, MAX_TRAIN_ROWS)
validation_eval_df = select_rows(validation_df, MAX_VALIDATION_ROWS)

print("train_eval", train_eval_df.shape)
print("validation_eval", validation_eval_df.shape)


train_eval (2000, 10)
validation_eval (400, 10)


In [ ]:
# Run validation first because it is the main model-selection split.
validation_records = run_split(validation_eval_df, "validation")


{'split': 'validation', 'target_rows': 400, 'already_done': 6, 'remaining': 394, 'batch_size': 30, 'path': '/content/output/qwen3_14b_zero_shot/qwen3_14b_zero_shot_strict_v2_validation.jsonl'}


validation:   0%|          | 0/14 [00:00<?, ?it/s]

In [ ]:
# Optional only: run train if you want a train/validation drift check later.
# For the current baseline, skip this cell and evaluate validation only.
# train_records = run_split(train_eval_df, "train")

## 8. 평가 지표 계산

이 평가는 raw zero-shot 기준선이다. calibration, threshold 조정, fallback 점수 대입을 하지 않는다. 따라서 파싱 실패율도 점수와 함께 봐야 한다.


In [ ]:
def records_to_pred_df(records: list[dict]) -> pd.DataFrame:
    rows = []
    for r in records:
        row = {
            "id": r["id"],
            "parse_ok": bool(r.get("parse_ok")),
            "parse_error": r.get("parse_error"),
            "elapsed_sec": r.get("elapsed_sec"),
        }
        parsed = r.get("parsed") if r.get("parse_ok") else None
        if parsed:
            for domain in DOMAINS:
                row[f"pred_{domain}"] = float(parsed[domain]["score"])
            row["pred_average"] = float(np.mean([row[f"pred_{d}"] for d in DOMAINS]))
        rows.append(row)
    pred_df = pd.DataFrame(rows)
    if not pred_df.empty:
        pred_df = pred_df.drop_duplicates("id", keep="last")
    return pred_df


def rmse(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def summarize_metrics(gold_df: pd.DataFrame, records: list[dict], split_name: str):
    pred_df = records_to_pred_df(records)
    merged = gold_df.merge(pred_df, on="id", how="left")
    expected = len(gold_df)
    generated = int(merged["parse_ok"].notna().sum()) if "parse_ok" in merged else 0
    parsed = merged[merged["parse_ok"] == True].copy()

    metric_rows = []
    for target in DOMAINS + ["average"]:
        y_true_col = f"gold_{target}"
        y_pred_col = f"pred_{target}"
        if len(parsed) == 0 or y_pred_col not in parsed:
            metric_rows.append({"split": split_name, "target": target, "rmse": np.nan, "spearman": np.nan, "n": 0})
            continue
        y_true = parsed[y_true_col].astype(float)
        y_pred = parsed[y_pred_col].astype(float)
        corr = spearmanr(y_true, y_pred).correlation if y_pred.nunique() > 1 else np.nan
        metric_rows.append({
            "split": split_name,
            "target": target,
            "rmse": rmse(y_true, y_pred),
            "spearman": float(corr) if corr is not None else np.nan,
            "n": len(parsed),
        })

    coverage = {
        "split": split_name,
        "expected_rows": expected,
        "generated_rows": generated,
        "parsed_rows": len(parsed),
        "missing_rows": expected - generated,
        "parse_fail_rows": generated - len(parsed),
        "parse_success_rate": len(parsed) / generated if generated else 0.0,
        "mean_elapsed_sec": float(pd.to_numeric(merged.get("elapsed_sec"), errors="coerce").mean()),
    }
    return pd.DataFrame(metric_rows), coverage, merged

validation_metrics, validation_coverage, validation_joined = summarize_metrics(
    validation_eval_df,
    validation_records,
    "validation",
)

print("coverage")
display(pd.DataFrame([validation_coverage]))

print("metrics")
display(validation_metrics.round(4))


coverage


,split,expected_rows,generated_rows,parsed_rows,missing_rows,parse_fail_rows,parse_success_rate,mean_elapsed_sec
0,validation,400,400,400,0,0,1.0,2.323237


metrics


,split,target,rmse,spearman,n
0,validation,content,0.7258,0.2820,400
1,validation,organization,0.8756,0.3982,400
2,validation,expression,0.9550,0.4473,400
3,validation,average,0.6667,0.4475,400


In [ ]:
# Save compact validation metric summaries for later comparison.
metrics_out = OUTPUT_DIR / f"{RUN_TAG}_metrics.csv"
coverage_out = OUTPUT_DIR / f"{RUN_TAG}_coverage.json"

validation_metrics.to_csv(metrics_out, index=False, encoding="utf-8-sig")
with coverage_out.open("w", encoding="utf-8") as f:
    json.dump({"validation": validation_coverage}, f, ensure_ascii=False, indent=2)

print(metrics_out)
print(coverage_out)

/content/output/qwen3_14b_zero_shot/qwen3_14b_zero_shot_strict_v2_metrics.csv
/content/output/qwen3_14b_zero_shot/qwen3_14b_zero_shot_strict_v2_coverage.json


## 9. 실패 사례와 샘플 확인

파싱 실패가 있으면 먼저 raw output을 보고 프롬프트를 조정한다. 점수 보정이나 fallback은 이 기준선 노트북에서는 적용하지 않는다.


In [ ]:
def show_failures(joined: pd.DataFrame, split_name: str, n: int = 5):
    fail = joined[(joined["parse_ok"].notna()) & (joined["parse_ok"] != True)].copy()
    print(split_name, "failures", len(fail))
    cols = ["id", "parse_error"]
    display(fail[cols].head(n))
    return fail.head(n)

validation_failures = show_failures(validation_joined, "validation")

validation failures 26


,id,parse_error
8,GWGR2300003530,missing object: content
23,GWGR2300010200,missing object: content
32,GWGR2300012710,missing object: content
35,GWGR2300012900,missing object: content
45,GWGR2300017300,missing object: content


In [ ]:
def show_prediction_samples(joined: pd.DataFrame, n: int = 5):
    cols = [
        "id", "prompt_num",
        "gold_content", "pred_content",
        "gold_organization", "pred_organization",
        "gold_expression", "pred_expression",
        "gold_average", "pred_average",
    ]
    available = [c for c in cols if c in joined.columns]
    display(joined[joined["parse_ok"] == True][available].head(n))

show_prediction_samples(validation_joined, n=10)


,id,prompt_num,gold_content,pred_content,gold_organization,pred_organization,gold_expression,pred_expression,gold_average,pred_average
0,GWGR2300001260,Q1,3.5,4.0,3.25,4.0,4.00,3.0,3.58,3.666667
1,GWGR2300001370,Q3,2.3,3.0,2.00,3.0,1.25,3.0,1.85,3.000000
2,GWGR2300002270,Q3,2.5,3.0,2.25,2.0,2.25,2.0,2.33,2.333333
3,GWGR2300002380,Q2,3.2,3.0,3.75,3.0,4.25,3.0,3.73,3.000000
4,GWGR2300002390,Q2,2.2,3.0,2.50,3.0,3.00,3.0,2.57,3.000000
5,GWGR2300002590,Q2,3.7,3.0,3.75,2.0,4.00,2.0,3.82,2.333333
6,GWGR2300003280,Q1,2.7,3.0,1.75,2.0,3.00,2.0,2.48,2.333333
7,GWGR2300003390,Q3,3.3,3.0,3.25,3.0,4.00,3.0,3.51,3.000000
9,GWGR2300003620,Q1,3.0,3.0,2.75,3.0,3.00,3.0,2.92,3.000000
10,GWGR2300003740,Q1,3.4,3.0,3.00,3.0,4.00,3.0,3.46,3.000000


In [ ]:
# Parse failure analysis
fail_df = validation_joined[
    (validation_joined["parse_ok"].notna()) & (validation_joined["parse_ok"] != True)
].copy()

print("failures:", len(fail_df))
display(fail_df["parse_error"].value_counts().head(20))

# raw output 확인
fail_ids = set(fail_df["id"])
fail_records = [r for r in validation_records if r["id"] in fail_ids]

for r in fail_records[:10]:
    print("=" * 80)
    print("id:", r["id"])
    print("error:", r["parse_error"])
    print(r["raw_output"][:2000])

failures: 26


parse_error
missing object: content    26
Name: count, dtype: int64

id: GWGR2300003530
error: missing object: content
{
  "content": {"score": 3, "rationale": "주제를 이해하고 있으며, 자신의 입장을 명확히 제시했으나, 반대 입장에 대한 논증이 부족하고, 고령자 운전 면허 제한에 대한 대안 제시가 미흡하여 논증의 깊이가 부족하다."},
  "organization": {"score": 3, "rationale": "서론, 본론, 결론의 구조는 갖추었으나, 본론 내부의 논점 전개가 약하고, 각 문단 간 연결성이 부족하여 논리 흐름이 약하다."},
  "expression": {"score": 3, "rationale": "전체적으로 문법 오류는 거의 없으나, 어휘 선택이 일반적이며, 표현의 자연스러움과 문장 흐름이 약간 부족하다.")
}
id: GWGR2300010200
error: missing object: content
{
  "content": {"score": 4, "rationale": "주제를 잘 이해하고 명확한 입장을 제시했으며, 혐오 표현의 법제화에 대한 두 가지 근거를 구체적으로 제시했으나, 논증의 깊이와 결론의 확신성에서 약간의 아쉬움이 있다."},
  "organization": {"score": 4, "rationale": "서론, 본론, 결론의 구조를 갖추고 있으며, 논점이 잘 전개되었으나, 일부 문단 간 연결성이 약간 부족하다."},
  "expression": {"score": 4, "rationale": "전체적으로 문법과 어휘 사용이 자연스럽고 정확하나, 일부 문장이 다소 복잡하거나 흐름이 끊어지는 경우가 있어 표현의 명확성에서 약간의 아쉬움이 있다.")
}
id: GWGR2300012710
error: missing object: content
{
  "content": {"score": 3, "rationale": "주제를 이해하고 있으나, 로봇세 도입에 대한 명확한 입장과 구체적인 근거가 부족하며, 논증의 깊이가 부족하다

In [ ]:
# Prediction vs gold distribution
parsed_df = validation_joined[validation_joined["parse_ok"] == True].copy()

for domain in DOMAINS:
    print("\n", domain)
    display(pd.DataFrame({
        "gold": parsed_df[f"gold_{domain}"].round(2),
        "pred": parsed_df[f"pred_{domain}"].round(2),
    }).describe().round(3))

    display(parsed_df[f"pred_{domain}"].value_counts().sort_index())


 content


,gold,pred
count,374.000,374.000
mean,3.233,3.380
std,0.646,0.533
min,1.100,2.000
25%,2.800,3.000
50%,3.200,3.000
75%,3.700,4.000
max,4.700,5.000


pred_content
2.0      8
3.0    217
4.0    148
5.0      1
Name: count, dtype: int64


 organization


,gold,pred
count,374.000,374.000
mean,3.284,3.198
std,0.879,0.742
min,1.000,1.000
25%,2.750,3.000
50%,3.250,3.000
75%,4.000,4.000
max,5.000,5.000


pred_organization
1.0      4
2.0     60
3.0    169
4.0    140
5.0      1
Name: count, dtype: int64


 expression


,gold,pred
count,374.000,374.000
mean,3.670,2.987
std,0.679,0.615
min,1.250,1.000
25%,3.250,3.000
50%,3.750,3.000
75%,4.000,3.000
max,5.000,5.000


pred_expression
1.0      4
2.0     60
3.0    248
4.0     61
5.0      1
Name: count, dtype: int64